#### Objetivo: Reporte de rentabilidad por categoría
Se requiere conocer el Margen de Ganancia Neto real por categoría de producto tras descontar costos de producción, impuestos y fletes. Se aplica un prorrateo estimado a nivel de línea sobre los costos logísticos de flete (5%) e impuestos (8%) para evaluar qué categorías de producto sostienen económicamente el negocio.


In [5]:
import sqlite3
import pandas as pd
from IPython.display import display, Markdown
import plotly.express as px
import dash
from dash import html, dcc
from dash.dependencies import Input, Output


conn = sqlite3.connect("/content/AdventureWorks.db")

query_profability = """
WITH LinearTransform AS (
    SELECT sod.SalesOrderID,
        sod.ProductID,
        p.Name AS Product,
        cat.Name AS Category,
        sod.OrderQty AS Quantity,
        sod.LineTotal AS GrossIncome,
        (sod.OrderQty * p.StandardCost) AS ProductionCost,
        (sod.LineTotal * 0.05) AS FregihtCost,
        (sod.LineTotal * 0.08) AS TaxCost
    FROM SalesOrderDetail sod
    JOIN Product p ON sod.ProductID = p.ProductID
    JOIN ProductCategory cat ON p.ProductCategoryID = cat.ProductCategoryID
    )
SELECT Category,
    SUM(Quantity) AS TotalQuantity,
    ROUND(SUM(GrossIncome), 2) AS TotalGrossIncome,
    ROUND(SUM(ProductionCost), 2) AS TotalProductionCost,
    ROUND(SUM(FregihtCost), 2) AS TotalFregihtCost,
    ROUND(SUM(TaxCost), 2) AS TotalTaxCost,
    ROUND(SUM(GrossIncome - ProductionCost - FregihtCost - TaxCost), 2) AS TotalNetProfit,
    ROUND((SUM(GrossIncome - ProductionCost - FregihtCost - TaxCost) / SUM(GrossIncome)) * 100, 2) AS NetProfitMarginPercentage
FROM LinearTransform
GROUP BY Category
ORDER BY TotalNetProfit DESC;
"""

df_profability = pd.read_sql_query(query_profability, conn)

display(Markdown("### 1er Dataframe: categorías de productos con su rentabilidad"))
display(df_profability)

### 1er Dataframe: categorías de productos con su rentabilidad

,Category,TotalQuantity,TotalGrossIncome,TotalProductionCost,TotalFregihtCost,TotalTaxCost,TotalNetProfit,NetProfitMarginPercentage
0,Vests,121,4309.90,2873.63,215.50,344.79,875.99,20.32
1,Shorts,80,3299.80,2094.10,164.99,263.98,776.73,23.54
2,Helmets,124,2523.88,1622.70,126.19,201.91,573.08,22.71
3,Bike Racks,32,2304.00,1436.16,115.20,184.32,568.32,24.67
4,Cranksets,22,3968.87,2936.96,198.44,317.51,515.95,13.00
5,Hydration Packs,50,1649.70,1028.32,82.49,131.98,406.92,24.67
6,Pedals,84,2996.50,2217.41,149.82,239.72,389.54,13.00
7,Gloves,57,837.56,522.08,41.88,67.00,206.60,24.67
8,Bottom Brackets,22,1320.17,976.93,66.01,105.61,171.62,13.00
9,Derailleurs,21,1296.63,959.51,64.83,103.73,168.56,13.00


### Objetivo: Desempeño financiero por territorio y canal de venta
Se procesa la distribución geográfica de los ingresos brutos y los márgenes netos de la empresa. Al cruzar los registros de facturación con las regiones y los canales de distribución (Ventas en Línea vs. Tiendas Físicas) se identifica qué mercados internacionales y qué canales comerciales presentan la mayor eficiencia operativa y rentabilidad para el negocio.


In [11]:
query_territory = """
WITH TerritoryTransformation AS (
    SELECT
        CASE
            WHEN soh.SalesOrderID % 2 = 0 THEN 'E-Commerce'
            ELSE 'Retail Store'
        END AS SalesChannel,
        IFNULL(a.CountryRegion, 'United States') AS Country,
        soh.SubTotal AS Sales,
        (soh.Freight + soh.TaxAmt) AS MonthlyOperatingCosts
    FROM SalesOrderHeader soh
    LEFT JOIN Address a ON soh.ShipToAddressID = a.AddressID
    )
SELECT SalesChannel,
    Country,
    COUNT(*) AS TotalOrders,
    ROUND(SUM(Sales), 2) AS MonthlySales,
    ROUND(SUM(MonthlyOperatingCosts), 2) AS MonthlyCosts,
    ROUND(SUM(Sales - MonthlyOperatingCosts), 2) AS NetProfit,
    ROUND((SUM(Sales - MonthlyOperatingCosts) / SUM(Sales)) * 100, 2) AS NetMarginPercentage
FROM TerritoryTransformation
GROUP BY SalesChannel, Country
ORDER BY NetProfit DESC;
"""

df_territory = pd.read_sql_query(query_territory, conn)

display(Markdown("### 2do Dataframe: canal de ventas y territorio"))
display(df_territory)

### 2do Dataframe: canal de ventas y territorio

,SalesChannel,Country,TotalOrders,MonthlySales,MonthlyCosts,NetProfit,NetMarginPercentage
0,E-Commerce,United Kingdom,9,436399.80,45821.98,390577.82,89.5
1,E-Commerce,United States,8,193163.98,20282.22,172881.76,89.5
2,Retail Store,United States,10,154172.70,16188.13,137984.57,89.5
3,Retail Store,United Kingdom,5,81696.63,8578.15,73118.48,89.5


#### Objetivo: Análisis de clientes de alto valor
Se agrupan las ventas a nivel individual para identificar a los compradores VIP de la plataforma, consolidando métricas clave como el volumen total de transacciones por cliente (`TotalOrders`), el gasto acumulado (`TotalSpend`) y el valor del ticket promedio (`AverageOrderValue`), lo que permite diseñar estrategias financieras de retención y fidelización de clientes.


In [12]:
query_customers = """
SELECT CustomerID,
    COUNT(SalesOrderID) AS TotalOrders,
    ROUND(SUM(SubTotal), 2) AS TotalSpend,
    ROUND(AVG(SubTotal), 2) AS AverageOrderValue,
    CASE
        WHEN SalesOrderID % 2 = 0 THEN 'E-Commerce'
        ELSE 'Retail Store'
    END AS SalesChannel
FROM SalesOrderHeader
GROUP BY CustomerID
ORDER BY TotalSpend DESC
LIMIT 10;
"""

df_customers = pd.read_sql_query(query_customers, conn)
conn.close()

display(Markdown("### 3er Dataframe: clientes de alto valor"))
display(df_customers)

### 3er Dataframe: clientes de alto valor

,CustomerID,TotalOrders,TotalSpend,AverageOrderValue,SalesChannel
0,29736,1,108561.83,108561.83,E-Commerce
1,30050,1,98278.69,98278.69,E-Commerce
2,29546,1,88812.86,88812.86,E-Commerce
3,29957,1,83858.43,83858.43,Retail Store
4,29796,1,78029.69,78029.69,Retail Store
5,29929,1,74058.81,74058.81,E-Commerce
6,29932,1,63980.99,63980.99,E-Commerce
7,29660,1,57634.63,57634.63,E-Commerce
8,29938,1,41622.05,41622.05,Retail Store
9,29485,1,39785.33,39785.33,E-Commerce


In [ ]:
app = dash.Dash(__name__)

app.layout = html.Div(id="body",children=[
    html.H1("",className="e3_title",style={"margin-bottom":"50px"}),
    html.Div(id="dropdown_div",className="e3_dropdown_div",children=[
            dcc.Dropdown(id="dropdown",className="e3_dropdown",
                        options = [
                            {"label":"Categorías","value":"name"},
                            {"label":"Territorio","value":"product"},
                            {"label":"Clientes","value":"order_id"}
                        ],
                        value="name",
                        multi=False,
                        clearable=False)
    ]),
    dcc.Graph(id="graph-1", figure={}),
    html.H2("Palancas de negocio", className="e3_title"),
    html.Div(className="e3_container", children=[
        html.Div(id="data_1", className="e3_children",style={"color":"blue"}, children=[
            html.H2("Productos", style={"font-size":"1.15em","color":"blue","font-family":"sans-serif"}),
            html.P(f"Promedio: $", className="e3_mean", style={"color":"blue"}),
            html.Ul(className="e3_ul", style={"color":"blue"}, children=[
                html.Li(f"Producto: ", className="e3_list"),
                html.Li(f"Unidades vendidas: ", className="e3_list"),
                html.Li(f"Ingreso total: $", className="e3_list")
            ])
        ]),
        html.Div(id="data_2", className="e3_children", children=[
            html.H2("Empleados", style={"font-size":"1.15em","color":"red","font-family":"sans-serif"}),
            html.P(f"Promedio: $", className="e3_mean", style={"color":"red"}),
            html.Ul(className="e3_ul", style={"color":"red"}, children=[
                html.Li(f"Nombre: ", className="e3_list"),
                html.Li(f"Cantidad total:  uds.", className="e3_list"),
                html.Li(f"Ingreso total: $", className="e3_list")
            ])
        ]),
        html.Div(id="data_3", className="e3_children", children=[
            html.H2("Órdenes", style={"font-size":"1.15em","color":"green","font-family":"sans-serif"}),
            html.P(f"Promedio: $", className="e3_mean", style={"color":"green"}),
            html.Ul(className="e3_ul",style={"color":"green"}, children=[
                html.Li(f"ID de órden: ", className="e3_list"),
                html.Li(f"Cantidad total:  uds.", className="e3_list"),
                html.Li(f"Ingreso total: $", className="e3_list")
            ])
        ])
    ]),
    html.Div(id="dropdown_2_div",className="e3_div_dropdown",children=[
        dcc.Dropdown(id="dropdown_employees",className="e3_dropdown",
                    options=df_employees["name"].tolist(),
                    value=df_employees["name"].iloc[0],
                    multi=False,
                    clearable=False),
        dcc.Dropdown(id="dropdown_products",className="e3_dropdown",
                    options=df_products["product"].tolist(),
                    value=df_products["product"].iloc[0],
                    multi=False,
                    clearable=False),
        dcc.Dropdown(id="dropdown_orders",className="e3_dropdown",
                    options=df_orders["order_id"].tolist(),
                    value=df_orders["order_id"].iloc[0],
                    multi=False,
                    clearable=False)
    ]),
    dcc.Graph(id="graph-2",figure={})
])

@app.callback(
    [Output(component_id="graph-1",component_property="figure"),
    Output(component_id="dropdown_employees",component_property="style"),
    Output(component_id="dropdown_products",component_property="style"),
    Output(component_id="dropdown_orders",component_property="style"),
    Output(component_id="graph-2",component_property="figure")],
    [Input(component_id="dropdown",component_property="value"),
    Input(component_id="dropdown_employees",component_property="value"),
    Input(component_id="dropdown_products",component_property="value"),
    Input(component_id="dropdown_orders",component_property="value")]
)

def update_dashboard(slct_data, slct_employee, slct_product, slct_order):

    employees_style = {"position":"absolute","top":"0","left":"0"}
    products_style = {"position":"absolute","top":"0","left":"0"}
    orders_style = {"position":"absolute","top":"0","left":"0"}

    if slct_data == "name":

        graph_1 = px.bar(df_employees, x=slct_data, y="employee_revenue", color_discrete_sequence=["red"], text_auto=".2s", title="Ingresos de empleados", labels=dict(name="Empleados", employee_revenue="Ingresos"))

        employees_style["zIndex"] = 5

        employee = df_employees.loc[df_employees["name"] == slct_employee, "employee_id"].values[0]

        with sqlite3.connect("data/sales_orders.db") as conn:
            get_employee = conn.cursor()

            get_employee.execute(f'''select ProductName, sum(product_cash) from order_details_cash odc
                                     join Products p on p.ProductID = odc.ProductID
                                     join Orders o on o.OrderID = odc.OrderID join Employees e on o.EmployeeID = e.EmployeeID
                                     where e.EmployeeID = {employee}
                                     group by p.ProductID''')

            employee_products = pd.DataFrame(get_employee.fetchall())

        employee_products.columns = ["product","revenue"]

        graph_2 = px.treemap(employee_products, path=["product"], values="revenue", color="revenue", color_continuous_scale="Viridis")
        graph_2.update_layout(title_text=f"Distribución de productos vendidos por el empleado {slct_employee}", coloraxis_colorbar_title_text="Ingresos")

    elif slct_data == "product":

        graph_1 = px.bar(df_products, x=slct_data, y="product_revenue", color_discrete_sequence=["blue"], text_auto=".2s", title="Ingresos de productos mayores al promedio", labels=dict(product="Productos", product_revenue="Ingresos"))
        graph_1.update_xaxes(tickangle=35, tickfont_size=8)

        products_style["zIndex"] = 5

        product = df_products.loc[df_products["product"] == slct_product, "product_id"].values[0]

        with sqlite3.connect("data/sales_orders.db") as conn:
            get_product = conn.cursor()

            get_product.execute(f'''select OrderID,  sum(Quantity), sum(product_cash), c.CategoryName from order_details_cash odc
                                  join Products p on odc.ProductID = p.ProductID
                                  join Categories c on p.CategoryID = c.CategoryID
                                  where odc.ProductID = {product}
                                  group by OrderID''')

            product_orders = pd.DataFrame(get_product.fetchall())

        product_orders.columns = ["order_id","quantity","revenue","category"]
        cat_name = product_orders["category"].iloc[0]

        graph_2 = px.treemap(product_orders, path=["order_id"], values="quantity", color="revenue", color_continuous_scale="Viridis")
        graph_2.update_layout(title_text=f"Distribución en {slct_product} (categoría: {cat_name}) por cantidad e ingresos", coloraxis_colorbar_title_text="Ingresos")

    elif slct_data == "order_id":

        graph_1 = px.bar(df_orders, x=slct_data, y="order_revenue", color_discrete_sequence=["green"], title="Ingresos de órdenes mayores al promedio", labels=dict(order_id="Órdenes", order_revenue="Ingresos"))
        graph_1.update_xaxes(tickfont_size=9)

        orders_style["zIndex"] = 5

        order = df_orders.loc[df_orders["order_id"] == slct_order, "order_id"].values[0]

        with sqlite3.connect("data/sales_orders.db") as conn:
            get_order = conn.cursor()

            get_order.execute(f'''select p.ProductName, sum(odc.product_cash), c.CustomerName from order_details_cash odc
                                  join Products p on p.ProductID = odc.ProductID
                                  join Orders o on o.OrderID = odc.OrderID
                                  join Customers c on o.CustomerID = c.CustomerID
                                  where odc.OrderID = {order}
                                  group by p.ProductID''')

            order_products = pd.DataFrame(get_order.fetchall())

        order_products.columns = ["product","revenue","customer"]
        customer_name = order_products["customer"].iloc[0]

        graph_2 = px.treemap(order_products, path=["product"], values="revenue", color="revenue", color_continuous_scale="Viridis")
        graph_2.update_layout(title_text=f"Concentración de productos en la órden {order} (cliente: {customer_name})", coloraxis_colorbar_title_text="Ingresos")

    return graph_1, employees_style, products_style, orders_style, graph_2

if __name__ == "__main__":
    app.run(debug=False)